In [ ]:
!pip install -q gwpy gwosc pycbc lalsuite fiestaEM torch torchvision scikit-learn matplotlib scipy numpy pandas tqdm

In [ ]:
import os, sys, json, warnings, zipfile, time, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from scipy.signal import welch
from scipy.interpolate import interp1d
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, confusion_matrix

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

RESULTS_DIR = Path('multimessenger_results')
DATA_DIR    = Path('mm_data_cache')
RESULTS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
SAMPLE_RATE    = 4096
SEGMENT_DUR    = 4
N_SAMPLES      = int(SAMPLE_RATE * SEGMENT_DUR)
F_LOW          = 20.0
F_HIGH         = 1024.0

N_GW_SIGNAL    = 2000
N_GW_NOISE     = 2000

EM_N_FILTERS   = 6
EM_N_TIMES     = 50
EM_FILTERS     = ['bessellux', 'bessellv', 'bessellb', 'bessellr', 'besselli', 'sdssg']
EM_T_GRID      = np.linspace(0.1, 10.0, EM_N_TIMES)

BATCH_SIZE     = 64
EPOCHS         = 80
LR             = 3e-4
WEIGHT_DECAY   = 1e-4
PATIENCE       = 18
LABEL_SMOOTH   = 0.05
MIXUP_ALPHA    = 0.3
DROPOUT_RATE   = 0.35
VAL_FRAC       = 0.15
TEST_FRAC      = 0.15

print('Configuration set.')

In [ ]:
from gwpy.timeseries import TimeSeries
from gwosc.datasets import event_gps

def fetch_gwosc_noise(detector, n_segments, seg_dur=SEGMENT_DUR, sample_rate=SAMPLE_RATE,
                      cache_file=None):
    if cache_file and cache_file.exists():
        print(f'  Loading cached noise: {cache_file}')
        return np.load(cache_file)

    gps_start = 1238166018
    total_needed = n_segments * seg_dur
    fetch_dur  = max(total_needed + 128, 4096)

    print(f'  Fetching {fetch_dur}s of {detector} O3a noise …')
    ts = TimeSeries.fetch_open_data(detector, gps_start, gps_start + fetch_dur,
                                     sample_rate=sample_rate, verbose=False, cache=True)
    data = ts.value.astype(np.float64)

    stride = int(seg_dur * sample_rate)
    segs = []
    for i in range(n_segments):
        s = i * stride
        segs.append(data[s:s + stride])

    segs = np.array(segs, dtype=np.float32)
    if cache_file:
        np.save(cache_file, segs)
    return segs

print('Fetching H1 noise …')
h1_noise = fetch_gwosc_noise('H1', N_GW_NOISE,
                              cache_file=DATA_DIR / 'h1_noise.npy')
print(f'  H1 noise shape: {h1_noise.shape}')

print('Fetching L1 noise …')
l1_noise = fetch_gwosc_noise('L1', N_GW_NOISE,
                              cache_file=DATA_DIR / 'l1_noise.npy')
print(f'  L1 noise shape: {l1_noise.shape}')
print('Noise fetch complete.')

In [ ]:
try:
    import lal
    import lalsimulation as lalsim
    HAS_LAL = True
except ImportError:
    HAS_LAL = False
    print('LAL not available; using PyCBC waveform generator instead.')

def generate_imrphenomd_strain(m1, m2, dist_mpc, sample_rate=SAMPLE_RATE,
                                seg_dur=SEGMENT_DUR, f_low=F_LOW):
    if HAS_LAL:
        m1_si = m1 * lal.MSUN_SI
        m2_si = m2 * lal.MSUN_SI
        dist_si = dist_mpc * 3.085677581491367e+22
        delta_t = 1.0 / sample_rate
        hp, hc = lalsim.SimInspiralChooseTDWaveform(
            m1_si, m2_si,
            0, 0, 0, 0, 0, 0,
            dist_si, 0, 0, 0, 0, 0,
            delta_t, f_low, f_low,
            lal.CreateDict(),
            lalsim.IMRPhenomD)
        hp_data = hp.data.data
    else:
        from pycbc.waveform import get_td_waveform
        hp, hc = get_td_waveform(
            approximant='IMRPhenomD',
            mass1=m1, mass2=m2,
            distance=dist_mpc,
            delta_t=1.0/sample_rate,
            f_lower=f_low)
        hp_data = hp.numpy()

    n_out = int(seg_dur * sample_rate)
    if len(hp_data) >= n_out:
        strain = hp_data[-n_out:].astype(np.float32)
    else:
        strain = np.zeros(n_out, dtype=np.float32)
        strain[-len(hp_data):] = hp_data
    return strain

def estimate_welch_psd(noise_segs, sample_rate=SAMPLE_RATE):
    all_f, all_p = [], []
    for seg in noise_segs[:min(200, len(noise_segs))]:
        f, p = welch(seg, fs=sample_rate, nperseg=sample_rate//2)
        all_f.append(f)
        all_p.append(p)
    return np.array(all_f[0]), np.median(np.array(all_p), axis=0)

def whiten_segment(data, psd_f, psd_p, sample_rate=SAMPLE_RATE):
    n = len(data)
    freqs = np.fft.rfftfreq(n, d=1.0/sample_rate)
    interp = interp1d(psd_f, psd_p, bounds_error=False,
                       fill_value=(psd_p[0], psd_p[-1]))
    psd_at_f = interp(freqs)
    psd_at_f = np.where(psd_at_f <= 0, 1e-45, psd_at_f)
    fd = np.fft.rfft(data)
    fd_w = fd / np.sqrt(psd_at_f * sample_rate / 2)
    mask = (freqs >= F_LOW) & (freqs <= F_HIGH)
    fd_w[~mask] = 0.0
    td_w = np.fft.irfft(fd_w, n=n).astype(np.float32)
    std = td_w.std()
    if std > 0:
        td_w /= std
    return td_w

print('Building PSD from H1 noise …')
psd_f, psd_p = estimate_welch_psd(h1_noise)
print(f'  PSD estimated over {len(psd_f)} frequency bins.')

In [ ]:
def synthetic_kilonova_lc(t_grid, m_ej, v_ej, kappa, dist_mpc,
                            filters=EM_FILTERS, noise_sigma=0.15):
    lcs = np.zeros((len(filters), len(t_grid)), dtype=np.float32)
    lambda_eff = np.array([3600, 5500, 4400, 6400, 8000, 4770], dtype=float)
    ref_lambda  = 5500.0

    Lbol_peak = 1e43 * (m_ej / 0.05) * (v_ej / 0.2) ** 2
    t_peak    = 0.98 * (kappa / 10.0) ** 0.5 * (m_ej / 0.05) ** 0.5 * \
                (v_ej / 0.2) ** (-0.5)
    t_peak    = max(t_peak, 0.3)

    tau     = t_grid / t_peak
    Lbol_t  = Lbol_peak * tau * np.exp(-0.5 * (tau - 1.0) ** 2 / 0.6 ** 2)
    Lbol_t  = np.where(Lbol_t < 0, 0, Lbol_t)

    T_color = 5000 * (1.0 + 3.0 / (t_grid + 0.5)) ** 0.5

    dist_cm = dist_mpc * 3.085677581491367e+24

    for fi, (filt, lam) in enumerate(zip(filters, lambda_eff)):
        x = (lam / ref_lambda) ** (-3)
        bb_weight = np.exp(-14387.77 / (lam * 1e-8 * T_color) + 14387.77 / (ref_lambda * 1e-8 * T_color))
        bb_weight = np.clip(bb_weight, 1e-10, 1e10)
        flux = Lbol_t * x * bb_weight / (4.0 * np.pi * dist_cm ** 2)
        with np.errstate(divide='ignore', invalid='ignore'):
            mag = np.where(flux > 0, -2.5 * np.log10(flux) - 48.6, 35.0)
        mag = np.clip(mag, 14.0, 34.0)
        mag += np.random.normal(0, noise_sigma, size=len(t_grid)).astype(np.float32)
        lcs[fi] = mag.astype(np.float32)

    return lcs

def synthetic_noise_lc(t_grid, filters=EM_FILTERS):
    lcs = np.zeros((len(filters), len(t_grid)), dtype=np.float32)
    for fi in range(len(filters)):
        base = np.random.uniform(22.0, 30.0)
        lcs[fi] = (base + np.random.normal(0, 0.5, size=len(t_grid))).astype(np.float32)
    return lcs

def normalize_lc(lc, t_grid=EM_T_GRID):
    lc = lc.copy()
    for fi in range(lc.shape[0]):
        mn, mx = lc[fi].min(), lc[fi].max()
        if mx > mn:
            lc[fi] = (lc[fi] - mn) / (mx - mn)
        else:
            lc[fi] = np.zeros_like(lc[fi])
    return lc

print('EM light curve generator ready.')

In [ ]:
gw_signal_cache  = DATA_DIR / 'gw_signals.npy'
gw_noise_cache   = DATA_DIR / 'gw_noise_w.npy'
em_signal_cache  = DATA_DIR / 'em_signals.npy'
em_noise_cache   = DATA_DIR / 'em_noise.npy'
labels_cache     = DATA_DIR / 'labels.npy'

all_caches = [gw_signal_cache, gw_noise_cache, em_signal_cache, em_noise_cache, labels_cache]

if all(c.exists() for c in all_caches):
    print('Loading cached dataset …')
    gw_signals   = np.load(gw_signal_cache)
    gw_noise_w   = np.load(gw_noise_cache)
    em_signals   = np.load(em_signal_cache)
    em_noise_lcs = np.load(em_noise_cache)
    labels       = np.load(labels_cache)
else:
    print('Generating GW signal dataset …')
    gw_signals = []
    for i in tqdm(range(N_GW_SIGNAL), desc='GW waveforms'):
        m1   = np.random.uniform(5, 80)
        m2   = np.random.uniform(5, min(m1, 80))
        dist = np.random.uniform(100, 1500)
        snr_target = np.random.uniform(8, 25)
        wf = generate_imrphenomd_strain(m1, m2, dist)
        noise_seg = h1_noise[i % len(h1_noise)]
        injected = noise_seg + wf
        w = whiten_segment(injected, psd_f, psd_p)
        gw_signals.append(w)
    gw_signals = np.array(gw_signals, dtype=np.float32)
    np.save(gw_signal_cache, gw_signals)
    print(f'  Saved GW signals: {gw_signals.shape}')

    print('Whitening noise segments …')
    gw_noise_w = []
    for i in tqdm(range(N_GW_NOISE), desc='Whitening noise'):
        w = whiten_segment(h1_noise[i], psd_f, psd_p)
        gw_noise_w.append(w)
    gw_noise_w = np.array(gw_noise_w, dtype=np.float32)
    np.save(gw_noise_cache, gw_noise_w)

    print('Generating EM light curves …')
    em_signals, em_noise_lcs = [], []
    for i in tqdm(range(N_GW_SIGNAL), desc='KN light curves'):
        m_ej  = np.random.uniform(0.005, 0.2)
        v_ej  = np.random.uniform(0.05, 0.4)
        kappa = np.random.choice([1.0, 5.0, 10.0, 30.0])
        dist  = np.random.uniform(50, 400)
        lc = synthetic_kilonova_lc(EM_T_GRID, m_ej, v_ej, kappa, dist)
        em_signals.append(normalize_lc(lc))
    for i in tqdm(range(N_GW_NOISE), desc='EM noise'):
        lc = synthetic_noise_lc(EM_T_GRID)
        em_noise_lcs.append(normalize_lc(lc))
    em_signals   = np.array(em_signals, dtype=np.float32)
    em_noise_lcs = np.array(em_noise_lcs, dtype=np.float32)
    np.save(em_signal_cache, em_signals)
    np.save(em_noise_cache, em_noise_lcs)

    labels = np.array([1]*N_GW_SIGNAL + [0]*N_GW_NOISE, dtype=np.float32)
    np.save(labels_cache, labels)
    print('Dataset generation complete.')

print(f'GW signals: {gw_signals.shape}, GW noise: {gw_noise_w.shape}')
print(f'EM signals: {em_signals.shape}, EM noise: {em_noise_lcs.shape}')
print(f'Labels: {labels.shape}')

In [ ]:
GW_ALL = np.concatenate([gw_signals, gw_noise_w], axis=0)
EM_ALL = np.concatenate([em_signals, em_noise_lcs], axis=0)

GW_ALL = GW_ALL[:, np.newaxis, :]

assert len(GW_ALL) == len(EM_ALL) == len(labels)

class MMDataset(Dataset):
    def __init__(self, gw, em, y):
        self.gw = torch.from_numpy(gw)
        self.em = torch.from_numpy(em)
        self.y  = torch.from_numpy(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.gw[idx], self.em[idx], self.y[idx]

full_ds = MMDataset(GW_ALL.astype(np.float32),
                    EM_ALL.astype(np.float32),
                    labels.astype(np.float32))

n_total = len(full_ds)
n_test  = int(n_total * TEST_FRAC)
n_val   = int(n_total * VAL_FRAC)
n_train = n_total - n_val - n_test

train_ds, val_ds, test_ds = random_split(
    full_ds, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {n_train}  Val: {n_val}  Test: {n_test}')

In [ ]:
class ResBlock1D(nn.Module):
    def __init__(self, ch, k=7, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, k, padding=k//2, bias=False),
            nn.BatchNorm1d(ch),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(ch, ch, k, padding=k//2, bias=False),
            nn.BatchNorm1d(ch),
        )
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.net(x))

class GWEncoder(nn.Module):
    def __init__(self, dropout=DROPOUT_RATE):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, 15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.GELU(),
            nn.Conv1d(32, 64, 9, stride=2, padding=4, bias=False),
            nn.BatchNorm1d(64), nn.GELU(),
        )
        self.blocks = nn.Sequential(
            ResBlock1D(64, k=7, dropout=dropout),
            nn.Conv1d(64, 128, 5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(128), nn.GELU(),
            ResBlock1D(128, k=5, dropout=dropout),
            nn.Conv1d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(256), nn.GELU(),
            ResBlock1D(256, k=3, dropout=dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.out_dim = 256

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return self.pool(x).squeeze(-1)


class EMEncoder(nn.Module):
    def __init__(self, n_filters=EM_N_FILTERS, n_times=EM_N_TIMES, dropout=DROPOUT_RATE):
        super().__init__()
        # Flatten filters into channel dim — avoids any spatial collapse issue
        self.proj = nn.Sequential(
            nn.Linear(n_filters * n_times, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
        )
        self.out_dim = 128

    def forward(self, x):
        # x: (B, n_filters, n_times)
        x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
        return self.proj(x.flatten(1))


class CrossModalAttention(nn.Module):
    def __init__(self, dim_gw=256, dim_em=128, heads=4):
        super().__init__()
        self.dim = dim_gw
        self.q = nn.Linear(dim_gw, dim_gw)
        self.k = nn.Linear(dim_em, dim_gw)
        self.v = nn.Linear(dim_em, dim_gw)
        self.heads = heads
        self.scale = (dim_gw // heads) ** -0.5
        self.out_proj = nn.Linear(dim_gw, dim_gw)

    def forward(self, gw_feat, em_feat):
        B = gw_feat.size(0)
        H = self.heads
        D = self.dim // H
        q = self.q(gw_feat).view(B, H, D)
        k = self.k(em_feat).view(B, H, D)
        v = self.v(em_feat).view(B, H, D)
        attn = (q * k).sum(-1, keepdim=True) * self.scale
        attn = torch.softmax(attn, dim=1)
        out  = (attn * v).view(B, self.dim)
        return self.out_proj(out)


class MMDetector(nn.Module):
    def __init__(self, dropout=DROPOUT_RATE):
        super().__init__()
        self.gw_enc  = GWEncoder(dropout=dropout)
        self.em_enc  = EMEncoder(dropout=dropout)
        self.cross   = CrossModalAttention(256, 128, heads=4)

        fused_dim = 256 + 128 + 256
        self.head = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, gw, em):
        gf = self.gw_enc(gw)
        ef = self.em_enc(em)
        ca = self.cross(gf, ef)
        fused = torch.cat([gf, ef, ca], dim=-1)
        return self.head(fused).squeeze(-1)


model = MMDetector().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MMDetector: {n_params:,} trainable parameters')

In [ ]:
class LabelSmoothBCE(nn.Module):
    def __init__(self, smooth=LABEL_SMOOTH):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        targets_s = targets * (1 - self.smooth) + 0.5 * self.smooth
        return F.binary_cross_entropy_with_logits(logits, targets_s)

def mixup_batch(gw, em, y, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return gw, em, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(gw.size(0), device=gw.device)
    gw2 = lam * gw + (1 - lam) * gw[idx]
    em2 = lam * em + (1 - lam) * em[idx]
    y2  = lam * y  + (1 - lam) * y[idx]
    return gw2, em2, y2

criterion = LabelSmoothBCE()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

print('Optimizer and loss configured.')

In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss, preds, trues = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for gw, em, y in loader:
            gw, em, y = gw.to(DEVICE), em.to(DEVICE), y.to(DEVICE)
            # Sanitise inputs
            gw = torch.nan_to_num(gw, nan=0.0, posinf=3.0, neginf=-3.0)
            em = torch.nan_to_num(em, nan=0.0, posinf=1.0, neginf=0.0)
            if train:
                gw, em, y = mixup_batch(gw, em, y)
                optimizer.zero_grad()
            logits = model(gw, em)
            logits = torch.nan_to_num(logits, nan=0.0)
            loss   = criterion(logits, y)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(y)
            preds.extend(torch.sigmoid(logits).detach().cpu().numpy())
            trues.extend(y.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    preds, trues = np.array(preds), np.array(trues)
    preds = np.nan_to_num(preds, nan=0.5)
    trues_bin = (trues >= 0.5).astype(int)
    auc = roc_auc_score(trues_bin, preds) if len(np.unique(trues_bin)) > 1 else 0.5
    return avg_loss, auc


history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}
best_val_loss = float('inf')
patience_cnt  = 0
best_ckpt     = RESULTS_DIR / 'best_model.pt'

print(f'Training for up to {EPOCHS} epochs (early stop patience={PATIENCE}) …')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_auc = run_epoch(train_loader, train=True)
    vl_loss, vl_auc = run_epoch(val_loader,   train=False)
    scheduler.step(epoch)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_auc'].append(tr_auc)
    history['val_auc'].append(vl_auc)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_cnt  = 0
        torch.save(model.state_dict(), best_ckpt)
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f'[{epoch:3d}/{EPOCHS}] '
              f'tr_loss={tr_loss:.4f} vl_loss={vl_loss:.4f} '
              f'tr_auc={tr_auc:.4f} vl_auc={vl_auc:.4f} '
              f'patience={patience_cnt}/{PATIENCE}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

elapsed = time.time() - t0
print(f'Training complete in {elapsed/60:.1f} min. Best val loss: {best_val_loss:.4f}')

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

all_preds, all_trues = [], []
with torch.no_grad():
    for gw, em, y in test_loader:
        gw = torch.nan_to_num(gw.to(DEVICE), nan=0.0, posinf=3.0, neginf=-3.0)
        em = torch.nan_to_num(em.to(DEVICE), nan=0.0, posinf=1.0, neginf=0.0)
        y  = y.to(DEVICE)
        logits = torch.nan_to_num(model(gw, em), nan=0.0)
        all_preds.extend(torch.sigmoid(logits).cpu().numpy())
        all_trues.extend(y.cpu().numpy())

all_preds     = np.nan_to_num(np.array(all_preds), nan=0.5)
all_trues     = np.array(all_trues)
all_trues_bin = (all_trues >= 0.5).astype(int)

test_auc = roc_auc_score(all_trues_bin, all_preds)
avg_prec = average_precision_score(all_trues_bin, all_preds)

fpr, tpr, thresh = roc_curve(all_trues_bin, all_preds)
tpr_at_1fpr  = float(np.interp(0.01,  fpr, tpr))
tpr_at_01fpr = float(np.interp(0.001, fpr, tpr))

prec, rec, _ = precision_recall_curve(all_trues_bin, all_preds)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_f1   = float(f1_scores.max())

print(f'Test ROC-AUC          : {test_auc:.4f}')
print(f'Avg Precision (AP)    : {avg_prec:.4f}')
print(f'TPR @ 1%  FPR         : {tpr_at_1fpr:.4f}')
print(f'TPR @ 0.1% FPR        : {tpr_at_01fpr:.4f}')
print(f'Best F1               : {best_f1:.4f}')

In [ ]:
def plot_training_curves(history, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a1a')
    for ax in axes:
        ax.set_facecolor('#0d1117')
        ax.tick_params(colors='#c9d1d9')
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')

    epochs_x = np.arange(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs_x, history['train_loss'], color='#58a6ff', lw=2, label='Train')
    axes[0].plot(epochs_x, history['val_loss'],   color='#ff7b72', lw=2, label='Val')
    axes[0].set_xlabel('Epoch', color='#c9d1d9')
    axes[0].set_ylabel('Loss',  color='#c9d1d9')
    axes[0].set_title('Loss Curves', color='#e6edf3', fontweight='bold')
    axes[0].legend(facecolor='#161b22', labelcolor='#c9d1d9')
    axes[0].grid(alpha=0.15, color='#30363d')

    axes[1].plot(epochs_x, history['train_auc'], color='#58a6ff', lw=2, label='Train')
    axes[1].plot(epochs_x, history['val_auc'],   color='#ff7b72', lw=2, label='Val')
    axes[1].set_xlabel('Epoch', color='#c9d1d9')
    axes[1].set_ylabel('AUC',   color='#c9d1d9')
    axes[1].set_title('AUC Curves', color='#e6edf3', fontweight='bold')
    axes[1].legend(facecolor='#161b22', labelcolor='#c9d1d9')
    axes[1].grid(alpha=0.15, color='#30363d')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close()


def plot_roc_pr(fpr, tpr, prec, rec, test_auc, avg_prec, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a1a')
    for ax in axes:
        ax.set_facecolor('#0d1117')
        ax.tick_params(colors='#c9d1d9')
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')

    axes[0].plot(fpr, tpr, color='#39d353', lw=2.5, label=f'AUC={test_auc:.4f}')
    axes[0].plot([0,1],[0,1],'--', color='#484f58', lw=1)
    axes[0].axvline(0.01,  color='#f78166', lw=1, ls=':', label='FPR=1%')
    axes[0].axvline(0.001, color='#d2a8ff', lw=1, ls=':', label='FPR=0.1%')
    axes[0].set_xlabel('FPR', color='#c9d1d9')
    axes[0].set_ylabel('TPR', color='#c9d1d9')
    axes[0].set_title('ROC Curve — GW+EM Detector', color='#e6edf3', fontweight='bold')
    axes[0].legend(facecolor='#161b22', labelcolor='#c9d1d9')
    axes[0].grid(alpha=0.15, color='#30363d')

    axes[1].plot(rec, prec, color='#58a6ff', lw=2.5, label=f'AP={avg_prec:.4f}')
    axes[1].set_xlabel('Recall',    color='#c9d1d9')
    axes[1].set_ylabel('Precision', color='#c9d1d9')
    axes[1].set_title('Precision-Recall Curve', color='#e6edf3', fontweight='bold')
    axes[1].legend(facecolor='#161b22', labelcolor='#c9d1d9')
    axes[1].grid(alpha=0.15, color='#30363d')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close()


def plot_score_distribution(preds, trues_bin, save_path):
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0a0a1a')
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#c9d1d9')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

    ax.hist(preds[trues_bin==1], bins=50, alpha=0.75, color='#39d353', label='Signal (GW+KN)', density=True)
    ax.hist(preds[trues_bin==0], bins=50, alpha=0.75, color='#f78166', label='Noise',          density=True)
    ax.axvline(0.5, color='white', lw=1.5, ls='--', label='Decision boundary')
    ax.set_xlabel('Detection Score', color='#c9d1d9')
    ax.set_ylabel('Density',         color='#c9d1d9')
    ax.set_title('Score Distribution — Joint GW+EM', color='#e6edf3', fontweight='bold')
    ax.legend(facecolor='#161b22', labelcolor='#c9d1d9')
    ax.grid(alpha=0.15, color='#30363d')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close()


def plot_sample_lc(em_signal, em_noise, save_path, t_grid=EM_T_GRID, filters=EM_FILTERS):
    colors = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a1a')
    for ax, lc, title in zip(axes,
                              [em_signal, em_noise],
                              ['Kilonova Light Curve (Signal)', 'Background LC (Noise)']):
        ax.set_facecolor('#0d1117')
        ax.tick_params(colors='#c9d1d9')
        for spine in ax.spines.values():
            spine.set_edgecolor('#30363d')
        for fi, filt in enumerate(filters):
            ax.plot(t_grid, lc[fi], color=colors[fi], lw=1.5, label=filt)
        ax.set_xlabel('Time (days)', color='#c9d1d9')
        ax.set_ylabel('Normalised Magnitude', color='#c9d1d9')
        ax.set_title(title, color='#e6edf3', fontweight='bold')
        ax.legend(facecolor='#161b22', labelcolor='#c9d1d9', ncol=2, fontsize=8)
        ax.grid(alpha=0.15, color='#30363d')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close()


plot_training_curves(history, RESULTS_DIR / 'training_curves.png')
plot_roc_pr(fpr, tpr, prec, rec, test_auc, avg_prec, RESULTS_DIR / 'roc_pr_curves.png')
plot_score_distribution(all_preds, all_trues_bin, RESULTS_DIR / 'score_distribution.png')
plot_sample_lc(em_signals[0], em_noise_lcs[0], RESULTS_DIR / 'sample_lightcurves.png')
print('All plots saved.')

In [ ]:
metrics = {
    'test_roc_auc':     round(float(test_auc),      4),
    'avg_precision_ap': round(float(avg_prec),       4),
    'tpr_at_1pct_fpr':  round(float(tpr_at_1fpr),   4),
    'tpr_at_0p1pct_fpr':round(float(tpr_at_01fpr),  4),
    'best_f1':          round(float(best_f1),        4),
    'epochs_trained':   len(history['train_loss']),
    'best_val_loss':    round(float(best_val_loss),  5),
    'n_params':         n_params,
    'train_samples':    n_train,
    'val_samples':      n_val,
    'test_samples':     n_test,
}

with open(RESULTS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame(history).to_csv(RESULTS_DIR / 'training_history.csv', index=False)

roc_df = pd.DataFrame({'fpr': fpr, 'tpr': tpr})
roc_df.to_csv(RESULTS_DIR / 'roc_curve.csv', index=False)

pred_df = pd.DataFrame({'score': all_preds, 'true_label': all_trues_bin})
pred_df.to_csv(RESULTS_DIR / 'test_predictions.csv', index=False)

torch.save(model.state_dict(), RESULTS_DIR / 'final_model.pt')

print('Metrics and artefacts saved.')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
zip_path = 'GW_EM_MM_DETECTOR_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in RESULTS_DIR.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(RESULTS_DIR.parent))

print(f'Zipped results → {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
    print('Auto-download triggered.')
except ImportError:
    print(f'Not in Colab — find your results at: {Path(zip_path).resolve()}')